# IDR vs Concentration Quadrant Chart ("Signature Chart")

**Purpose:**  


> *"Dependency on one axis, concentration on the other, bubble size by import  
> volume, colour by composite score. The upper-right quadrant IS the finding,  
> visible without reading a number."*

---

## What This Chart Shows

| Quadrant | Dependency | Concentration | Interpretation |
|---|---|---|---|
| **Upper-right** | High | High | Maximum risk: depends heavily on imports AND those come from few suppliers |
| Upper-left | Low | High | Concentrated but self-sufficient: low urgency |
| Lower-right | High | Low | Dependent but diversified: resilient to single-supplier shock |
| Lower-left | Low | Low | Self-sufficient and diversified: lowest risk |

Countries in the **upper-right quadrant** are the headline finding.

---

## Data Requirements

This notebook needs **two inputs**:
1. `data/cleaned/fbs_cleaned.csv` — for import dependency ratio (IDR) and  
   caloric data. Produced by `build_fbs_dataset.py`.
2. `data/cleaned/trade_matrix_cleaned.csv` — for computing concentration  
   (HHI, Shannon) directly if the concentration CSV is not yet available.

If `fbs_cleaned.csv` is not available (requires FAOSTAT FBS download), the  
notebook falls back to **import intensity** as a proxy for dependency.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", font_scale=1.05)

# ---- Paths ----
ROOT = Path(".").resolve().parent
TRADE_MATRIX = ROOT / "data" / "cleaned" / "trade_matrix_cleaned.csv"
FBS_PATH = ROOT / "data" / "cleaned" / "fbs_cleaned.csv"
CONC_PATH = ROOT / "data" / "cleaned" / "concentration_with_shannon.csv"
VIZ_DIR = ROOT / "visualizations"
OUTPUT_DIR = ROOT / "data" / "cleaned"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# ---- Item-to-commodity mapping ----
ITEM_MAP = {
    "Wheat": "Wheat",
    "Wheat and meslin flour": "Wheat",
    "Rice, paddy (rice milled equivalent)": "Rice",
    "Rice, milled": "Rice",
    "Maize (corn)": "Maize",
}

COMMODITIES = ["Wheat", "Rice", "Maize"]

# ---- Analysis window (3-year mean per methodology spec) ----
WINDOW_START = 2021
WINDOW_END = 2023

print(f"FBS available: {FBS_PATH.exists()}")
print(f"Concentration CSV available: {CONC_PATH.exists()}")
print(f"Trade matrix available: {TRADE_MATRIX.exists()}")

## 1. Build Concentration Metrics

If the Shannon notebook output (`concentration_with_shannon.csv`) exists, load it.  
Otherwise, compute HHI and Shannon from the trade matrix directly.

In [ ]:
if CONC_PATH.exists():
    # ---- Use pre-computed concentration from Shannon notebook ----
    print(f"Loading pre-computed concentration from {CONC_PATH.name}")
    conc = pd.read_csv(CONC_PATH)
else:
    # ---- Compute from trade matrix ----
    print("Concentration CSV not found. Computing from trade_matrix_cleaned.csv...")
    raw = pd.read_csv(TRADE_MATRIX)
    imp = raw[
        (raw["Element"] == "Import quantity")
        & (raw["Unit"] == "t")
        & (raw["Value"] > 0)
    ].copy()
    imp["Commodity"] = imp["Item"].map(ITEM_MAP)
    imp = imp[imp["Commodity"].notna()].copy()

    GROUP = ["Reporter Country Code", "Reporter Countries", "Commodity", "Year"]
    FLOW = GROUP + ["Partner Country Code", "Partner Countries"]

    flows = (
        imp.groupby(FLOW, as_index=False)["Value"]
        .sum()
        .rename(columns={"Value": "import_t"})
    )
    tot = flows.groupby(GROUP, as_index=False)["import_t"].sum().rename(
        columns={"import_t": "total_t"}
    )
    flows = flows.merge(tot, on=GROUP, how="left")
    flows["share"] = flows["import_t"] / flows["total_t"]

    def shannon_h(s):
        p = s[s > 0].values
        return -np.sum(p * np.log(p)) if len(p) > 1 else 0.0

    conc = flows.groupby(GROUP, as_index=False).agg(
        partner_hhi=("share", lambda x: (x ** 2).sum()),
        shannon_h=("share", shannon_h),
        partner_count=("Partner Country Code", "nunique"),
        top_partner_share=("share", "max"),
        total_import_quantity_t=("import_t", "sum"),
    )
    conc["effective_suppliers_hhi"] = 1.0 / conc["partner_hhi"]
    conc["effective_suppliers_shannon"] = np.exp(conc["shannon_h"])

# ---- Filter to analysis window and average across years ----
conc_w = conc[conc["Year"].between(WINDOW_START, WINDOW_END)].copy()

conc_avg = (
    conc_w.groupby(
        ["Reporter Country Code", "Reporter Countries", "Commodity"],
        as_index=False,
    ).agg(
        mean_hhi=("partner_hhi", "mean"),
        mean_shannon_h=("shannon_h", "mean"),
        mean_top_share=("top_partner_share", "mean"),
        mean_eff_suppliers_hhi=("effective_suppliers_hhi", "mean"),
        mean_import_t=("total_import_quantity_t", "mean"),
    )
)

print(f"Concentration averages: {len(conc_avg)} country-commodity groups")
conc_avg.head(3)

## 2. Build Dependency Metric (IDR or Fallback)

The **import dependency ratio** (IDR) is the proper measure:  
`IDR = imports / (production + imports - exports + stock_variation)`

If FBS data is available, use it. Otherwise, compute **import intensity** as  
a proxy:  
`import_intensity = imports / (imports + exports)`

Import intensity is NOT dependency (it says nothing about domestic production),  
but it separates real importers from exporters and re-export hubs.

In [ ]:
USE_FBS = FBS_PATH.exists()

if USE_FBS:
    # ---- Load FBS and extract IDR ----
    print("Using FBS-based import dependency ratio (IDR)")
    fbs = pd.read_csv(FBS_PATH)
    fbs = fbs[fbs["Year"].between(WINDOW_START, WINDOW_END)].copy()
    
    dep = (
        fbs.groupby(["Area Code", "Area", "Commodity"], as_index=False)
        .agg(
            mean_idr=("import_dependency_ratio", "mean"),
            mean_kcal=("kcal_per_capita_day", "mean"),
        )
    )
    dep = dep.rename(columns={"Area Code": "Reporter Country Code", "Area": "Reporter Countries"})
    dependency_col = "mean_idr"
    dependency_label = "Import Dependency Ratio (IDR)"
    
else:
    # ---- Fallback: compute import intensity from trade matrix ----
    # import_intensity = imports / (imports + exports)
    print("FBS not available. Falling back to import intensity proxy.")
    print("NOTE: Import intensity is NOT dependency — it ignores domestic production.")
    
    raw = pd.read_csv(TRADE_MATRIX) if "raw" not in dir() else raw
    trade_agg = raw[
        (raw["Unit"] == "t")
        & (raw["Value"] > 0)
        & raw["Year"].between(WINDOW_START, WINDOW_END)
    ].copy()
    trade_agg["Commodity"] = trade_agg["Item"].map(ITEM_MAP)
    trade_agg = trade_agg[trade_agg["Commodity"].notna()].copy()
    
    # Aggregate imports and exports per reporter-commodity-year
    imp_total = (
        trade_agg[trade_agg["Element"] == "Import quantity"]
        .groupby(["Reporter Country Code", "Reporter Countries", "Commodity", "Year"],
                 as_index=False)["Value"]
        .sum()
        .rename(columns={"Value": "imports_t"})
    )
    exp_total = (
        trade_agg[trade_agg["Element"] == "Export quantity"]
        .groupby(["Reporter Country Code", "Reporter Countries", "Commodity", "Year"],
                 as_index=False)["Value"]
        .sum()
        .rename(columns={"Value": "exports_t"})
    )
    
    # Merge and compute intensity
    intensity = imp_total.merge(
        exp_total,
        on=["Reporter Country Code", "Reporter Countries", "Commodity", "Year"],
        how="left",
    )
    intensity["exports_t"] = intensity["exports_t"].fillna(0)
    intensity["import_intensity"] = (
        intensity["imports_t"]
        / (intensity["imports_t"] + intensity["exports_t"]).replace(0, np.nan)
    )
    
    dep = (
        intensity.groupby(
            ["Reporter Country Code", "Reporter Countries", "Commodity"],
            as_index=False,
        ).agg(mean_idr=("import_intensity", "mean"))
    )
    dependency_col = "mean_idr"
    dependency_label = "Import Intensity (proxy for IDR)"

print(f"Dependency metric: {dependency_label}")
print(f"Dependency rows: {len(dep)}")

## 3. Join Dependency + Concentration

In [ ]:
# ---- Merge dependency with concentration ----
merged = conc_avg.merge(
    dep,
    on=["Reporter Country Code", "Reporter Countries", "Commodity"],
    how="inner",
)

# ---- Drop rows with missing data ----
merged = merged.dropna(subset=[dependency_col, "mean_hhi"]).copy()

# ---- Compute a simple composite risk score for colouring ----
# Normalise both to [0,1] and use weighted geometric mean (matching calculate_risk.py)
EPSILON = 1e-3

for commodity in COMMODITIES:
    mask = merged["Commodity"] == commodity
    for col in [dependency_col, "mean_hhi"]:
        sub = merged.loc[mask, col]
        lo, hi = sub.quantile(0.05), sub.quantile(0.95)
        clipped = sub.clip(lower=lo, upper=hi)
        if hi > lo:
            merged.loc[mask, f"{col}_norm"] = (clipped - lo) / (hi - lo)
        else:
            merged.loc[mask, f"{col}_norm"] = 0.0

merged["risk_score"] = (
    (merged[f"{dependency_col}_norm"] + EPSILON) ** 0.5
    * (merged["mean_hhi_norm"] + EPSILON) ** 0.5
)

# ---- Exclude net exporters (import intensity < 0.5 in fallback mode) ----
# Countries that are net exporters rank high on concentration by accident
# (small imports from a single neighbour). This is explained in
# wheat_exposure_firstcut.py.
if not USE_FBS:
    n_before = len(merged)
    merged = merged[merged[dependency_col] > 0.5].copy()
    print(f"Excluded {n_before - len(merged)} net exporters (intensity <= 0.5)")

print(f"Merged dataset: {len(merged)} country-commodity groups")
merged.head()

## 4. The Signature Chart — IDR vs HHI Quadrant

This is the chart. Dependency on the x-axis, concentration on the y-axis,  
bubble size proportional to import volume, colour by risk score.  
Upper-right is the danger zone.

In [ ]:
def quadrant_chart(data, commodity, dep_col, dep_label, conc_col, conc_label,
                   size_col, color_col, ax, annotate_top_n=8):
    """
    Draw a single quadrant chart for one commodity.
    
    Parameters
    ----------
    data : pd.DataFrame
        Merged data for one commodity.
    dep_col : str
        Column for x-axis (dependency).
    conc_col : str
        Column for y-axis (concentration).
    size_col : str
        Column for bubble size.
    color_col : str
        Column for bubble colour.
    ax : matplotlib.axes.Axes
        Axes to draw on.
    annotate_top_n : int
        Number of highest-risk points to label.
    """
    sub = data.copy()
    
    # Scale bubble sizes: sqrt of volume, normalised to a visible range
    sizes = np.sqrt(sub[size_col]) / np.sqrt(sub[size_col].max()) * 400 + 15
    
    scatter = ax.scatter(
        sub[dep_col], sub[conc_col],
        s=sizes,
        c=sub[color_col],
        cmap="YlOrRd",
        alpha=0.7,
        edgecolors="grey",
        linewidths=0.4,
    )
    
    # Quadrant lines at medians
    med_x = sub[dep_col].median()
    med_y = sub[conc_col].median()
    ax.axhline(med_y, color="grey", linestyle="--", linewidth=0.7, alpha=0.5)
    ax.axvline(med_x, color="grey", linestyle="--", linewidth=0.7, alpha=0.5)
    
    # Label the quadrants
    ax.text(0.98, 0.98, "HIGH RISK", transform=ax.transAxes,
            ha="right", va="top", fontsize=8, color="red", alpha=0.6, weight="bold")
    ax.text(0.02, 0.02, "LOW RISK", transform=ax.transAxes,
            ha="left", va="bottom", fontsize=8, color="green", alpha=0.6, weight="bold")
    
    # Annotate highest-risk countries
    top = sub.nlargest(annotate_top_n, color_col)
    for _, row in top.iterrows():
        # Truncate long country names
        name = row["Reporter Countries"]
        if len(name) > 20:
            name = name[:18] + "..."
        ax.annotate(
            name,
            (row[dep_col], row[conc_col]),
            fontsize=7, alpha=0.8,
            textcoords="offset points", xytext=(5, 5),
        )
    
    ax.set_xlabel(dep_label)
    ax.set_ylabel(conc_label)
    ax.set_title(commodity, fontsize=13, weight="bold")
    
    return scatter


# ---- Draw the triptych: one panel per commodity ----
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

for ax, commodity in zip(axes, COMMODITIES):
    sub = merged[merged["Commodity"] == commodity]
    if sub.empty:
        ax.set_title(f"{commodity} (no data)")
        continue
    scatter = quadrant_chart(
        data=sub,
        commodity=commodity,
        dep_col=dependency_col,
        dep_label=dependency_label,
        conc_col="mean_hhi",
        conc_label="HHI (supplier concentration)",
        size_col="mean_import_t",
        color_col="risk_score",
        ax=ax,
    )

# Shared colorbar
cbar = fig.colorbar(scatter, ax=axes, shrink=0.6, pad=0.02)
cbar.set_label("Risk Score", fontsize=11)

fig.suptitle(
    f"Import Dependency vs Supplier Concentration ({WINDOW_START}-{WINDOW_END} mean)\n"
    "Bubble size = import volume | Colour = risk score | Upper-right = highest risk",
    fontsize=14, y=1.04,
)
plt.tight_layout()
plt.savefig(str(VIZ_DIR / "idr_vs_hhi_quadrant.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Companion Chart: IDR vs Shannon H

Same structure but with Shannon diversity on the y-axis. Here, the **lower-right**  
quadrant is the danger zone (high dependency, LOW diversity).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

for ax, commodity in zip(axes, COMMODITIES):
    sub = merged[merged["Commodity"] == commodity]
    if sub.empty:
        ax.set_title(f"{commodity} (no data)")
        continue
    
    # For Shannon, lower = more concentrated = more risk.
    # Flip the risk interpretation for the scatter: annotate LOWEST Shannon.
    sizes = np.sqrt(sub["mean_import_t"]) / np.sqrt(sub["mean_import_t"].max()) * 400 + 15
    scatter = ax.scatter(
        sub[dependency_col], sub["mean_shannon_h"],
        s=sizes,
        c=sub["risk_score"],
        cmap="YlOrRd",
        alpha=0.7,
        edgecolors="grey",
        linewidths=0.4,
    )
    
    med_x = sub[dependency_col].median()
    med_y = sub["mean_shannon_h"].median()
    ax.axhline(med_y, color="grey", linestyle="--", linewidth=0.7, alpha=0.5)
    ax.axvline(med_x, color="grey", linestyle="--", linewidth=0.7, alpha=0.5)
    
    # Lower-right is high risk for Shannon
    ax.text(0.98, 0.02, "HIGH RISK", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=8, color="red", alpha=0.6, weight="bold")
    ax.text(0.02, 0.98, "LOW RISK", transform=ax.transAxes,
            ha="left", va="top", fontsize=8, color="green", alpha=0.6, weight="bold")
    
    # Annotate highest-risk (highest risk_score) countries
    top = sub.nlargest(8, "risk_score")
    for _, row in top.iterrows():
        name = row["Reporter Countries"]
        if len(name) > 20:
            name = name[:18] + "..."
        ax.annotate(
            name, (row[dependency_col], row["mean_shannon_h"]),
            fontsize=7, alpha=0.8,
            textcoords="offset points", xytext=(5, -8),
        )
    
    ax.set_xlabel(dependency_label)
    ax.set_ylabel("Shannon H (supplier diversity)")
    ax.set_title(commodity, fontsize=13, weight="bold")

cbar = fig.colorbar(scatter, ax=axes, shrink=0.6, pad=0.02)
cbar.set_label("Risk Score", fontsize=11)

fig.suptitle(
    f"Import Dependency vs Shannon Diversity ({WINDOW_START}-{WINDOW_END} mean)\n"
    "Bubble size = import volume | Colour = risk score | Lower-right = highest risk",
    fontsize=14, y=1.04,
)
plt.tight_layout()
plt.savefig(str(VIZ_DIR / "idr_vs_shannon_quadrant.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Upper-Right Quadrant Countries (Risk Table)

In [ ]:
# ---- Extract countries in the upper-right quadrant (above median on both axes) ----
for commodity in COMMODITIES:
    sub = merged[merged["Commodity"] == commodity].copy()
    if sub.empty:
        continue
    
    med_dep = sub[dependency_col].median()
    med_hhi = sub["mean_hhi"].median()
    
    # Upper-right: above median dependency AND above median concentration
    upper_right = sub[
        (sub[dependency_col] > med_dep)
        & (sub["mean_hhi"] > med_hhi)
    ].sort_values("risk_score", ascending=False)
    
    print(f"\n{'='*60}")
    print(f"{commodity}: UPPER-RIGHT QUADRANT (high dependency + high concentration)")
    print(f"{'='*60}")
    print(f"Dependency median: {med_dep:.3f} | HHI median: {med_hhi:.3f}")
    print(f"Countries in upper-right: {len(upper_right)}\n")
    
    display_cols = [
        "Reporter Countries", dependency_col, "mean_hhi",
        "mean_shannon_h", "mean_eff_suppliers_hhi",
        "mean_top_share", "mean_import_t", "risk_score",
    ]
    print(
        upper_right[display_cols].head(20).to_string(
            index=False,
            float_format="{:.3f}".format,
            formatters={"mean_import_t": "{:,.0f}".format},
        )
    )

## 7. Save Output

In [ ]:
out_path = OUTPUT_DIR / "quadrant_analysis.csv"
merged.to_csv(out_path, index=False)
print(f"Saved quadrant analysis: {out_path}")
print(f"Rows: {len(merged)}")

print("\nVisualizations saved:")
print("  - visualizations/idr_vs_hhi_quadrant.png")
print("  - visualizations/idr_vs_shannon_quadrant.png")